# 3. Structural Analysis

This chapter surveys the per-frame structural analyses that return NumPy arrays:
**RMSD**, **radius of gyration (Rg)**, **distance RMSD (DRMS)**, geometric
measurements via **trj_analysis**, and the **average structure** (avecrd). All run
on the bundled BPTI trajectory.


In [ ]:
import numpy as np
from genepie import genesis_exe, SMolecule
from genepie.tests.conftest import BPTI_PDB, BPTI_PSF, BPTI_DCD

mol = SMolecule.from_file(pdb=BPTI_PDB, psf=BPTI_PSF, ref=BPTI_PDB)
trajs, subset = genesis_exe.crd_convert(
    mol, trj_files=[str(BPTI_DCD)], trj_format="DCD",
    trj_type="COOR+BOX", selection="all",
)
traj = trajs[0]
print("frames:", traj.nframe)

## RMSD and radius of gyration

In [ ]:
rmsd = genesis_exe.rmsd_analysis(
    mol, traj, analysis_selection="an:CA",
    fitting_selection="an:CA", fitting_method="TR+ROT",
).rmsd

rg = genesis_exe.rg_analysis(
    mol, traj, analysis_selection="an:CA", mass_weighted=True,
).rg

print(f"RMSD: mean={rmsd.mean():.3f} A   Rg: mean={rg.mean():.3f} A")

## Distance RMSD (DRMS)

DRMS compares intramolecular contact *distances* to a reference, so it needs no
structural fitting. We build a simple C&alpha; contact list from the reference
coordinates (contacts between 1-6 &#8491;, excluding near-sequence neighbours).


In [ ]:
def ca_contacts(mol, ca_idx, lo=1.0, hi=6.0, exclude=4):
    ref = mol.atom_refcoord
    resno = mol.residue_no
    pairs, dists = [], []
    for a in range(len(ca_idx)):
        for b in range(a + 1, len(ca_idx)):
            i, j = ca_idx[a], ca_idx[b]
            if abs(resno[i - 1] - resno[j - 1]) < exclude:
                continue
            d = np.linalg.norm(ref[i - 1] - ref[j - 1])
            if lo <= d < hi:
                pairs.append([min(i, j), max(i, j)])
                dists.append(d)
    return np.array(pairs, dtype=np.int32).T, np.array(dists, dtype=np.float64)

ca_idx = genesis_exe.selection(mol, "an:CA")
contact_list, contact_dist = ca_contacts(mol, ca_idx)
print(f"{contact_list.shape[1]} reference contacts")

drms = genesis_exe.drms_analysis(
    traj, contact_list=contact_list, contact_dist=contact_dist,
    ana_period=1, pbc_correct=False,
).drms
print(f"DRMS: mean={drms.mean():.3f} A")

## Geometric measurements with `trj_analysis`

`trj_analysis` computes distances, angles, and dihedrals from atom-index tuples
(1-indexed). Here we track a C&alpha;-C&alpha; distance and a C&alpha; pseudo-angle.


In [ ]:
ca1 = genesis_exe.selection(mol, "rno:1 and an:CA")[0]
ca2 = genesis_exe.selection(mol, "rno:2 and an:CA")[0]
ca3 = genesis_exe.selection(mol, "rno:3 and an:CA")[0]

geo = genesis_exe.trj_analysis(
    traj,
    distance_pairs=np.array([[ca1, ca2]], dtype=np.int32),
    angle_triplets=np.array([[ca1, ca2, ca3]], dtype=np.int32),
)
print("distance shape:", geo.distance.shape, " angle shape:", geo.angle.shape)
print(f"CA1-CA2 distance: mean={geo.distance[:, 0].mean():.2f} A")
print(f"CA1-CA2-CA3 angle: mean={geo.angle[:, 0].mean():.1f} deg")

## Plot the scalar time series together

In [ ]:
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go
pio.renderers.default = "notebook"

def tint(hex_color, alpha=0.12):
    h = hex_color.lstrip("#")
    return f"rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{alpha})"

fig = make_subplots(rows=2, cols=2, vertical_spacing=0.13, horizontal_spacing=0.09,
                    subplot_titles=("C&alpha; RMSD", "Radius of gyration",
                                    "DRMS", "CA1-CA2 distance"))
panels = [
    (rmsd,               "#4C72B0", 1, 1, "RMSD (&#8491;)"),
    (rg,                 "#DD8452", 1, 2, "Rg (&#8491;)"),
    (drms,               "#55A868", 2, 1, "DRMS (&#8491;)"),
    (geo.distance[:, 0], "#C44E52", 2, 2, "distance (&#8491;)"),
]
for y, color, r, c, ytitle in panels:
    fig.add_trace(go.Scatter(y=y, mode="lines", line=dict(color=color, width=2.5),
                             fill="tozeroy", fillcolor=tint(color)),
                  row=r, col=c)
    fig.update_yaxes(title_text=ytitle, row=r, col=c)
    fig.update_xaxes(title_text="Frame", row=r, col=c)
fig.update_layout(height=580, showlegend=False, template="plotly_white",
                  font=dict(family="Inter, Helvetica, Arial, sans-serif", size=12, color="#333"),
                  margin=dict(l=60, r=30, t=70, b=50),
                  title=dict(text="<b>BPTI structural analyses</b>", font=dict(size=18)))
fig

## Average structure (avecrd)

`avecrd_analysis` iteratively fits the trajectory and averages the coordinates,
returning the mean structure as a PDB string.


In [ ]:
ave = genesis_exe.avecrd_analysis(
    mol, traj, selection_group=["an:CA"], fitting_method="TR+ROT",
    fitting_atom=1, analysis_atom=1, check_only=False, num_iterations=5,
)
print(ave.pdb[:240], "...")